In [1]:
import os
os.chdir('/home/smallyan/eval_agent')

# Load environment variables from .bashrc
import subprocess
result = subprocess.run(['bash', '-c', 'source /home/smallyan/.bashrc && env'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if '=' in line:
        key, _, value = line.partition('=')
        os.environ[key] = value

# Set model cache directory
os.environ['HF_HOME'] = '/net/projects2/chai-lab/shared_models'
os.environ['TRANSFORMERS_CACHE'] = '/net/projects2/chai-lab/shared_models'

print(f"Working directory: {os.getcwd()}")
print(f"HF_HOME: {os.environ.get('HF_HOME')}")

Working directory: /home/smallyan/eval_agent
HF_HOME: /net/projects2/chai-lab/shared_models


# Circuit Analysis Code Evaluation

This notebook evaluates the code implementation for circuit analysis in the repository located at `/net/scratch2/smallyan/InterpDetect_eval`.

## Objective
Evaluate all code blocks in the InterpDetect implementation for:
1. Runnable (Y/N) - Does the code execute without error?
2. Correct-Implementation (Y/N) - Does the logic implement the described computation correctly?
3. Redundant (Y/N) - Does the block duplicate another's computation?
4. Irrelevant (Y/N) - Does the block contribute to the project goal?

## Repository Structure
Based on the CodeWalkthrough.md:
- **Part 1: Preprocessing Pipeline** - preprocess.py, generate_response_gpt.py, generate_labels.py, filter.py, helper.py
- **Part 2: Training & Prediction** - compute_scores.py, classifier.py, predict.py
- **Part 3: Baseline Comparisons** - run_gpt.py, run_groq.py, run_hf.py, run_ragas.py, run_refchecker.py, run_trulens.py

In [2]:
# Initialize evaluation tracking
evaluation_results = []

def record_eval(script_name, block_id, block_desc, runnable, correct, redundant, irrelevant, note=""):
    """Record evaluation result for a block"""
    evaluation_results.append({
        'script': script_name,
        'block_id': block_id,
        'description': block_desc,
        'runnable': runnable,
        'correct': correct,
        'redundant': redundant,
        'irrelevant': irrelevant,
        'note': note
    })

corrections_made = 0
blocks_that_failed = 0

print("Evaluation tracking initialized")

Evaluation tracking initialized


## Part 2: Core Analysis Code (Training & Prediction)

### 1. compute_scores.py Evaluation

In [3]:
# Block 1: compute_scores.py - Import statements
try:
    import torch
    from transformers import AutoTokenizer
    from transformer_lens import HookedTransformer
    import json
    from torch.nn import functional as F
    from typing import Dict, List, Tuple
    from sentence_transformers import SentenceTransformer
    import numpy as np
    import pandas as pd
    import argparse
    import sys
    import os
    import gc
    from tqdm import tqdm
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy.stats import pointbiserialr
    
    record_eval("compute_scores.py", "B1", "Import statements", "Y", "Y", "N", "N")
    print("Block 1 (Imports): SUCCESS")
except Exception as e:
    record_eval("compute_scores.py", "B1", "Import statements", "N", "N", "N", "N", str(e))
    print(f"Block 1 (Imports): FAILED - {e}")

/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Block 1 (Imports): SUCCESS


In [4]:
# Block 2: compute_scores.py - load_examples function
def load_examples(file_path):
    """Load examples from JSONL file"""
    print(f"Loading examples from {file_path}...")
    
    try:
        examples = []
        with open(file_path, 'r') as f:
            for line in f:
                data = json.loads(line)
                examples.append(data)
        
        print(f"Loaded {len(examples)} examples")
        return examples
    except Exception as e:
        print(f"Error loading examples: {e}")
        return []

# Test the function
test_path = "/net/scratch2/smallyan/InterpDetect_eval/scripts/preprocess/datasets/test/test1176_w_labels_filtered.jsonl"
try:
    examples = load_examples(test_path)
    if len(examples) > 0:
        print(f"First example keys: {list(examples[0].keys())}")
        record_eval("compute_scores.py", "B2", "load_examples function", "Y", "Y", "N", "N")
    else:
        record_eval("compute_scores.py", "B2", "load_examples function", "N", "N", "N", "N", "No examples loaded")
except Exception as e:
    record_eval("compute_scores.py", "B2", "load_examples function", "N", "N", "N", "N", str(e))

Loading examples from /net/scratch2/smallyan/InterpDetect_eval/scripts/preprocess/datasets/test/test1176_w_labels_filtered.jsonl...
Loaded 256 examples
First example keys: ['id', 'question', 'documents', 'documents_sentences', 'prompt', 'prompt_spans', 'num_tokens', 'response', 'response_spans', 'labels', 'hallucinated_llama-4-maverick-17b-128e-instruct', 'hallucinated_gpt-oss-120b', 'labels_llama', 'labels_gpt']


In [5]:
# Block 3: compute_scores.py - setup_models function
def setup_models(model_name, hf_model_name, device="cuda"):
    """Setup tokenizer, model, and sentence transformer"""
    print(f"Setting up models: {model_name}, {hf_model_name}")
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
        
        model = HookedTransformer.from_pretrained(
            model_name,
            device="cpu",
            torch_dtype=torch.float16
        )
        model.to(device)
        
        bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5").to(device)
        
        return tokenizer, model, bge_model
    except Exception as e:
        print(f"Error setting up models: {e}")
        return None, None, None

# Test setup_models
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

try:
    tokenizer, model, bge_model = setup_models("qwen3-0.6b", "Qwen/Qwen3-0.6B", device)
    if model is not None:
        print(f"Model loaded successfully on {next(model.parameters()).device}")
        print(f"Model config: n_layers={model.cfg.n_layers}, n_heads={model.cfg.n_heads}")
        record_eval("compute_scores.py", "B3", "setup_models function", "Y", "Y", "N", "N")
    else:
        record_eval("compute_scores.py", "B3", "setup_models function", "N", "N", "N", "N", "Model setup returned None")
except Exception as e:
    record_eval("compute_scores.py", "B3", "setup_models function", "N", "N", "N", "N", str(e))
    print(f"Error: {e}")

Using device: cuda
Setting up models: qwen3-0.6b, Qwen/Qwen3-0.6B


In [6]:
# Check model loading status
print(f"Model loaded: {model is not None}")
if model is not None:
    print(f"Device: {next(model.parameters()).device}")
    print(f"Config: n_layers={model.cfg.n_layers}, n_heads={model.cfg.n_heads}, n_ctx={model.cfg.n_ctx}")

In [7]:
# Continue checking
print("Checking status...")